In [1]:
from script.OCR import ocr
import cv2
from ultralytics import YOLO
import glob
import os
from paddleocr import PaddleOCR
import numpy as np

j:\Python\MVP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### YOLO Model

In [ ]:
model = YOLO("models/best_obb.pt")

test_images = glob.glob("test_images/*.jpg") + glob.glob("test_images/*.png")
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    save=True,
    project=r"J:\Python\MVP\outputs",
    name="predict_obb_best",
    exist_ok=True,
    conf=0.5,
    iou=0.4,
)

### Cropping

In [ ]:
output_dir = r"J:\Python\MVP\outputs\crops_obb"
os.makedirs(output_dir, exist_ok=True)


def order_points(pts):
    """
    Order points as:
    top-left, top-right, bottom-right, bottom-left
    """
    pts = np.array(pts, dtype=np.float32)

    rect = np.zeros((4, 2), dtype=np.float32)

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # Top-left
    rect[2] = pts[np.argmax(s)]      # Bottom-right

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # Top-right
    rect[3] = pts[np.argmax(diff)]   # Bottom-left

    return rect



for img_path, result in zip(test_images, results):
    img = cv2.imread(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    obb_corners = result.obb.xyxyxyxy.cpu().numpy()

    for i, corners in enumerate(obb_corners):

        # -------------------------------
        # Reorder the corners correctly
        # -------------------------------
        corners = order_points(corners)


        # -------------------------------
        # Compute width
        # -------------------------------
        widthA = np.linalg.norm(corners[2] - corners[3])
        widthB = np.linalg.norm(corners[1] - corners[0])
        width = int(max(widthA, widthB))

        # -------------------------------
        # Compute height
        # -------------------------------
        heightA = np.linalg.norm(corners[1] - corners[2])
        heightB = np.linalg.norm(corners[0] - corners[3])
        height = int(max(heightA, heightB))

        if width < 2 or height < 2:
            continue

        dst = np.array([
            [0, 0],
            [width - 1, 0],
            [width - 1, height - 1],
            [0, height - 1]
        ], dtype=np.float32)


        # -------------------------------
        # Perspective transform
        # -------------------------------
        M = cv2.getPerspectiveTransform(corners, dst)

        crop = cv2.warpPerspective(img, M, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

        # -------------------------------
        # Rotate if vertical
        # -------------------------------
        # h, w = crop.shape[:2]
        # if h > w:
        #     crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)

        

        crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
        cv2.imwrite(crop_filename, crop)

### Preprocessing

In [ ]:
paths = glob.glob("outputs/crops_obb/*.jpg")
os.makedirs("outputs/pre_obb", exist_ok=True)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
target_height = 256
max_width = 1000
max_scale = 3

for path in paths:
    img = cv2.imread(path)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE
    gray = clahe.apply(gray)

    # Resizing
    h, w = gray.shape[:2]

    scale = target_height / h
    if scale > max_scale:
        scale = max_scale

    new_w = int(w * scale)
    new_h = int(h * scale)

    if new_w > max_width:
        scale = max_width / w
        new_w = max_width
        new_h = int(h * scale)


    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC

    resized = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=interpolation)

    filename = os.path.basename(path)
    cv2.imwrite(os.path.join("outputs/pre_obb", filename), resized)

### Paddle OCR

In [2]:
# Initialize PaddleOCR
paddle_ocr = PaddleOCR(
    enable_mkldnn=False,
    use_doc_orientation_classify=True,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

j:\Python\MVP\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('P

#### Single Book

In [3]:
path = 'outputs/pre_obb/img2_spine5.jpg'

img = cv2.imread(path)
ocr_results = paddle_ocr.predict(img)

img_name = os.path.splitext(os.path.basename(path))[0]

print(f'Image: {img_name}')

for res in ocr_results:
    texts = res['rec_texts']
    scores = res['rec_scores']

    for text, score in zip(texts, scores):
        print(f'Text: {text:30} Confidence: {score:.4f}')

Image: img2_spine5
Text: MARK                           Confidence: 1.0000
Text: THE SUBTLE ART OF              Confidence: 0.9999
Text: 子                              Confidence: 0.2651
Text: MANSON                         Confidence: 1.0000
Text: NOT GIVING A FPCK              Confidence: 0.9620


#### Multiple Books

In [17]:
CONFIDENCE_THRESHOLD = 0.8

In [18]:
paths = glob.glob("outputs/pre_obb/*.jpg")

filtered_results = []

for path in paths:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            if score >= CONFIDENCE_THRESHOLD:
                filtered_results.append({'image': img_name, 'text': text, 'confidence': score})
                print(f'Text: {text:30} Confidence: {score:.4f}')

Image: img1_spine0
Image: img1_spine1
Image: img1_spine2
Image: img1_spine3
Image: img1_spine4
Image: img1_spine5
Text: QASIM ALI SHAH                 Confidence: 0.9999
Text: FOUNDATION                     Confidence: 1.0000
Image: img2_spine0
Text: Yuval                          Confidence: 0.8565
Text: Noah                           Confidence: 0.9878
Text: Hararl                         Confidence: 0.9293
Text: Homo Deus                      Confidence: 0.8999
Image: img2_spine1
Text: HARRYPOTTER                    Confidence: 0.9971
Text: HELOSOER'SSTIRE                Confidence: 0.8326
Text: IK.ROWLING                     Confidence: 0.9532
Image: img2_spine10
Text: THE ALCHEMIST                  Confidence: 0.9998
Text: PauloCoelho                    Confidence: 0.9971
Image: img2_spine2
Text: Nexus                          Confidence: 1.0000
Text: Yuval Noah Harari              Confidence: 1.0000
Image: img2_spine3
Text: BIG MAGIC                      Confidence: 0.9993
Text: 

In [19]:
for path in paths:
    img = cv2.imread(path)
    results = paddle_ocr.predict(img)

    keep_img = True

    for res in results:
        if any(score < CONFIDENCE_THRESHOLD for score in res['rec_scores']):
            keep_img = False
            break

    if keep_img:
        print(os.path.basename(path))

img1_spine0.jpg
img1_spine1.jpg
img2_spine1.jpg
img2_spine10.jpg
img2_spine2.jpg
img2_spine3.jpg
img2_spine6.jpg
img2_spine8.jpg
IMG_20260722_090255_spine5.jpg
IMG_20260722_090305_spine10.jpg
IMG_20260722_090305_spine11.jpg
IMG_20260722_090305_spine12.jpg
IMG_20260722_090305_spine2.jpg
IMG_20260722_090305_spine3.jpg
IMG_20260722_090305_spine5.jpg
IMG_20260722_090305_spine6.jpg
IMG_20260722_090305_spine7.jpg
IMG_20260722_090305_spine9.jpg
urdu_spine10.jpg
urdu_spine4.jpg
urdu_spine8.jpg


### Class

In [1]:
from script.OCR import ocr
import cv2
from ultralytics import YOLO
import glob
import os
from paddleocr import PaddleOCR
import numpy as np

j:\Python\MVP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from script.pipeline import SpinePipeline

pl = SpinePipeline()

j:\Python\MVP\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('P

In [3]:
img = cv2.imread('test_images/img2.jpg')
pl.results(img)

[{'text': ['Yuval', 'Noah', 'Harart', 'Homo Deus'],
  'scores': [0.8694530725479126,
   0.9823206067085266,
   0.6799725890159607,
   0.8810722231864929]},
 {'text': ['/HARRYPOTTER', 'PRLOSOPIER SITONE', 'JK.ROWLING', 'ELIOAL'],
  'scores': [0.9809632897377014,
   0.5948059558868408,
   0.9832955598831177,
   0.3438164293766022]},
 {'text': ['Nexus', 'Yuval Noah Harari'],
  'scores': [0.9999945759773254, 0.9998606443405151]},
 {'text': ['BIG MAGIC', 'ELIZABETH GILBERT'],
  'scores': [0.9982133507728577, 0.9747300148010254]},
 {'text': ['MARK', 'THE SUBTLE ART OF', '子', 'MANSON', 'NOT GIVING A FPCK'],
  'scores': [0.9999898672103882,
   0.9999212622642517,
   0.7975640296936035,
   0.9999383091926575,
   0.9701559543609619]},
 {'text': ['ISLAM: ITS MEANING AND MESSAGE'], 'scores': [0.9828366041183472]},
 {'text': ['Friedrich Nietzsche', 'Beyond Good and Evi', 'REAONCE'],
  'scores': [0.9834322929382324, 0.9199519753456116, 0.6811272501945496]},
 {'text': ['ZERO To ONE', 'PETER THIEL'],
